# LangChain과 Chroma를 활용한 RAG 구성

1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
    - 토큰수 초과로 답변을 생성하지 못할 수도 있고
    - 문서가 길면 (인풋이 길면) 답변 생성이 오래걸림
    - split 된 데이터 chunk를 Large Language Model(LLM)에게 전달하면 토큰 절약 가능
    - 비용 감소와 답변 생성시간 감소의 효과
3. 임베딩 --> 벡터 DB에 저장
4. 질문이 있을 때, 벡터 DB에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

## 1. 문서 쪼개서 document_list 생성
- 매뉴얼 정보 추가하기
- 마크다운 파일 1개를 1개의 document 객체로 만들어 리스트 반환 (문서 쪼개는 것 대신)

In [2]:
import os
import json
import re
from langchain.schema import Document # langchain_core.documents 로 변경 가능

# 2. 교육자료 로드 함수 (사용자님이 작성하신 코드)
def load_documents_from_metadata_json(
    md_dir="md_pdf",
    metadata_filepath="all_metadata.json"
):
    with open(metadata_filepath, "r", encoding="utf-8") as f:
        all_metadata = json.load(f)

    docs = []
    for metadata in all_metadata:
        md_file_name = metadata.get("source")
        if not md_file_name:
            continue

        file_path = os.path.join(md_dir, md_file_name)
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
            
            docs.append(
                Document(
                    page_content=content,
                    metadata=metadata
                )
            )
        except FileNotFoundError:
            print(f"⚠️  경고: {file_path} 파일을 찾을 수 없습니다.")
            continue
            
    print(f"✅ 교육자료 Document 개수: {len(docs)}")
    return docs

# 3. 교육자료만 불러오기
education_docs = load_documents_from_metadata_json(
    md_dir="../../data/finance/processed/md_pdf",
    metadata_filepath="../../data/finance/json/all_metadata.json"
)

✅ 교육자료 Document 개수: 49


- hwp 불러오기

In [8]:
from langchain_core.documents import Document
import json

def load_docs_from_json(input_path="../../data/finance/json/split_hwp.json"):
    """
    JSON 파일을 langchain_core.documents.Document 리스트로 복원.
    """
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    document_list = [
        Document(page_content=item["page_content"], metadata=item["metadata"])
        for item in data
    ]
    print(f"✅ {len(document_list)}개의 문서를 불러왔습니다.")
    return document_list

In [9]:
document_list = load_docs_from_json("../../data/finance/json/split_hwp.json")

✅ 34개의 문서를 불러왔습니다.


In [10]:
document_list[4]

Document(metadata={'origin_pdf': '정산지침', 'page_num': '5조. 자금관리 및 집행'}, page_content="5조.  자금관리 및 집행4.  공용카드 사용 가. 공용카드에 의한 정산은 본사로부터 승인받은 신용카드, 체크카드 및 특정목적 전용카드(주유 등 특정 목적을 위해 발급한 카드)로 지급한 비용만 정산 가능하다. 조직망은 신용카드를 우선으로 사용하되 현지 여건 상 신용카드 발급이 불가능한 경우, 본사의 사전승인을 얻어 체크카드를 발급받을 수 있다. 단, 불가피한 사유로 인하여 공무수행을 위한 경비를 개인카드로 지급한 경우에는 동 사유를 명기하여 정산할 수 있다. 나. 공용카드를 사적인 용도로 사용할 수 없다. 다. 신용카드 및 체크카드 사용료를 정산할 때는 지출세부내역(청구서 등)과 영수증(카드슬립 등)을 첨부하여 카드사용일자에 건별로 정산하여야 한다. 단, 지급금액이 건당 미화 50달러 이하인 건은 ‘VIII. 비목별 세부 정산요령 14. 소액경비 및 다발성 경비’ 내용에 따라 정산할 수 있다. 별도의 지출세부내역 수령이 불가능하고 영수증상 지출내역이 없는 경우, 별도로 세부내역을 기재하여야 한다. 라. 불가피하게 주말, 공휴일, 휴가기간 및 심야(23시이후)에 사용한 경우에는 그 사유를 증빙상에 명기하여야 한다. 마. 신용카드 및 체크카드를 신규로 발급하는 경우에는 본사로부터 사전에 승인을 얻어야 하며, 신규/교체발급 또는 명의변경을 하는 경우 그 즉시 내역을 본사로 보고하고, ERP재무회계시스템에 등록해야 한다. 참고로 인사발령 등으로 관장 또는 총무 교체 시 카드 명의 변경을 하는 것은 신규발급이 아니라 교체발급임 바. 신용카드 및 체크카드 최대보유한도는 본사 파견직원 수에 따라 1인은 1개, 2인은 2개, 3인은 3개, 4인은 3개, 5인이상은 본사 파견직원 수 × 70%(소수점 이하 절사)이며, 부득이한 사유로 한도를 초과하여 보유코자 할 경우 그 사유를 보고하고 사전에 본사 승인을 득하여야 한다. 사.

In [6]:
education_docs[4]

Document(metadata={'source': '005.md', 'origin_pdf': '교육자료', 'page_num': 5, 'images': ['005_img0.png']}, page_content="1.1.4.3. ERP <자금잔액명세서> 상의 '경리장부' 잔액\n\n  \n\n그리고 ERP<자금잔액명세서> 메뉴에서 <나. 은행잔고대사표-경리장부잔액> 컬럼을 보면 각 장부별 잔액이 있는데, 이 수치는 직접 기재하는 것이 아니고  \nERP<경리장부조회> 에서의 각 장부 잔액이 자동으로 표시됩니다. (신용카드의 경우 결제가 되지 않은 경우 (-)잔액이 정상)\n\n  \n\n다만, 이 컬럼에 자동으로 기재된 금액이 ERP <경리장부조회> 에서의 각 장부 잔액과 다른 금액으로  \n표시되는 경우가 있는데, 이 케이스는 1.1.4.5.를 참고하세요\n\n1.1.4.4. 차액\n===========\n\n  \n\n결국 위(1.1.4.1)(1.1.4.2)에서 직접 기재한 <나. 은행잔고대사표-은행잔고증명서-현지화금액/송금통화금액/신용카드금액> 상의 금액과, (1.1.4.3)에서 자동으로  \n표시되는 <나. 은행잔고대사표-경리장부잔액> 상의 금액간의 차이금액이 <나. 은행잔고대사표-차액> 에 자동으로 표시 되게 됩니다.  \n이 차액이 왜 나게 되는지를 각장부별 <다.차액설명>에 직접 기재해야하고, 여기서 기재한 차액설명의 금액 합이 <나.은행잔고대사표-차액>과 일치해야지만  \nERP <자금잔액명세서>가 정상적으로 확정이 됩니다.\n\n· ERP 자금잔액명세서 경리장부조회 경리장부잔액")

## 2. 임베딩

In [2]:
# 3. 임베딩 해주기 --> 백터DB에 저장 (Chroma 쓸꺼고 / 인메모리라 간단)
from dotenv import load_dotenv # 임베딩에 openai key 인자가 있어서 환경변수를 로드해줘야 함
from langchain_openai import OpenAIEmbeddings 

# 환경변수 불러옴
load_dotenv()

# OpenAI에서 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large') # 임베딩 모델을 large로 바꿔주기

## 3. 벡터DB 생성 (Pinecone) 

In [14]:
import os

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'finance-index-v2' # finance-new-index : 정산지침+교육자료(매뉴얼ppt)
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

### Namespace와 Chunk_id 생성

In [32]:
# 2. page_num에서 '숫자'만 추출하는 함수 (예: "5조. 자금관리" -> "5", "35" -> "35")
def extract_only_numbers(page_text):
    if page_text is None:
        return "0"
    page_str = str(page_text).strip()
    
    # 문자열에서 처음 등장하는 숫자들만 싹 가져옵니다.
    match = re.search(r'\d+', page_str)
    if match:
        return match.group(0)
    return "0"

# 3. 100% 영문+숫자(ASCII) 조합으로 순차 ID를 생성하는 함수
def prepare_ascii_sequential_ids(docs, prefix_code):
    processed_ids = []
    page_chunk_counts = {}
    
    for doc in docs:
        page_num = doc.metadata.get("page_num", "0")
        
        # 숫자만 깔끔하게 추출 (예: "5조" -> "5")
        clean_num = extract_only_numbers(page_num)
        
        # 그룹핑 키 생성 (예: "cal_5" 또는 "edu_35")
        group_key = f"{prefix_code}_{clean_num}"
        
        if group_key not in page_chunk_counts:
            page_chunk_counts[group_key] = 1
        else:
            page_chunk_counts[group_key] += 1
            
        # 🌟 최종 ASCII chunk_id 규칙: {cal/edu}_{조/쪽수}_{순차넘버링}
        # 예: cal_5_1, cal_5_2, edu_35_1
        chunk_id = f"{group_key}_{page_chunk_counts[group_key]}"
        processed_ids.append(chunk_id)
        
    return processed_ids

In [33]:
# =====================================================================
# 4. [정산지침] 데이터 처리 및 업로드 (Namespace: calculation_guide)
# =====================================================================
print("⏳ 정산지침 문서 영문 ASCII ID 생성 중...")
# 정산지침은 앞글자를 'cal'로 지정
hwp_ids = prepare_ascii_sequential_ids(document_list, prefix_code="cal")

print(f"정산지침 ID 변환 확인 (상위 5개): {hwp_ids[:5]}")
# 출력 예시: ['cal_1_1', 'cal_1_2', 'cal_2_1', ...]

print("🚀 'cal_guide' Namespace로 Pinecone 빌드 시작...")
database_cal = PineconeVectorStore.from_documents(
    documents=document_list,
    embedding=embedding,
    index_name=index_name,
    namespace="cal_guide",  # Namespace 영문 유지
    ids=hwp_ids                     # ASCII ID 주입
)
print("✅ 정산지침 문서 업로드 완료!\n")

⏳ 정산지침 문서 영문 ASCII ID 생성 중...
정산지침 ID 변환 확인 (상위 5개): ['cal_1_1', 'cal_3_1', 'cal_4_1', 'cal_5_1', 'cal_5_2']


In [41]:
# =====================================================================
# 5. [교육자료] 데이터 처리 및 업로드 (Namespace: education_material)
# =====================================================================
print("⏳ 교육자료 문서 영문 ASCII ID 생성 중...")
# 교육자료는 앞글자를 'edu'로 지정
edu_ids = prepare_ascii_sequential_ids(education_docs, prefix_code="edu")

print(f"교육자료 ID 변환 확인 (상위 5개): {edu_ids[:5]}")
# 출력 예시: ['edu_5_1', 'edu_5_2', ...]

print("🚀 'edu_material' Namespace로 Pinecone 빌드 시작...")
database_edu = PineconeVectorStore.from_documents(
    documents=education_docs,
    embedding=embedding,
    index_name=index_name,
    namespace="edu_material",  # Namespace 영문 유지
    ids=edu_ids                      # ASCII ID 주입
)
print("✅ 교육자료 문서 업로드 완료!")

⏳ 교육자료 문서 영문 ASCII ID 생성 중...
교육자료 ID 변환 확인 (상위 5개): ['edu_1_1', 'edu_2_1', 'edu_3_1', 'edu_4_1', 'edu_5_1']


In [18]:
# ==========================================
# 🔥 2. [여기에 위치합니다] 통합 database 객체 생성
# ==========================================
database = PineconeVectorStore.from_existing_index(
    index_name=index_name, 
    embedding=embedding
)
print("✅ 전체 인덱스를 통합 조회할 수 있는 'database' 객체가 준비되었습니다.")

✅ 전체 인덱스를 통합 조회할 수 있는 'database' 객체가 준비되었습니다.


In [8]:
'''
# 존재하는 인덱스를 완전 삭제
if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)
    print(f"Deleted index: {index_name}") 
'''

Deleted index: finance-new-index


In [4]:
''' 
# 데이터를 추가(insert)할 때는 `from_documents()`
# 데이터를 추가한 이후에는 `from_existing_index()`를 사용 (데이터 추가없이 검색만)
#database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name) 
database = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embedding) 
'''

In [6]:
'''
# 교육자료 추가 업데이트
print("🚀 교육자료 업로드를 시작합니다... (잠시만 기다려주세요)")
database.add_documents(education_docs)
print("🎉 교육자료 업로드가 완료되었습니다!")
'''

🚀 교육자료 업로드를 시작합니다... (잠시만 기다려주세요)
🎉 교육자료 업로드가 완료되었습니다!


In [19]:
import os
from pinecone import Pinecone

pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

# 1️⃣ 인덱스 상태 확인
index_info = pc.describe_index(index_name)
print(f"📊 인덱스 상태: {index_info.status}")

# 2️⃣ 인덱스 통계(벡터 개수) 확인
index_stats = pc.Index(index_name).describe_index_stats()
print(f"📦 '{index_name}' 인덱스 요약:")
print(index_stats)

# 3️⃣ 깔끔하게 개수만 출력하고 싶다면
vector_count = index_stats["total_vector_count"]
print(f"✅ 현재 '{index_name}'에는 {vector_count:,}개의 벡터가 저장되어 있습니다.") # 83개 + 28= 111

📊 인덱스 상태: {'ready': True, 'state': 'Ready'}
📦 'finance-index-v2' 인덱스 요약:
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '276',
                                    'content-type': 'application/json',
                                    'date': 'Wed, 17 Jun 2026 06:51:01 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '46',
                                    'x-pinecone-request-latency-ms': '46',
                                    'x-pinecone-response-duration-ms': '47'}},
 'dimension': 3072,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'cal_guide': {'vector_count': 34},
                'edu_material': {'vector_count': 49},
                'public_official_conflict_interest_act': {'vector_count': 28}},
 'storageFullness': 0.0,
 'total_vector

## ESG 매뉴얼 추가

In [30]:
# 메타 데이터 확인
import json

# 1. 경로 설정
metadata_filepath = "../../data/esg/json/KOTRA 임직원 행동강령.json"
target_index = 59  # 5번째 청크 (인덱스 4)

try:
    with open(metadata_filepath, "r", encoding="utf-8") as f:
        esg_data = json.load(f)
    
    if len(esg_data) > target_index:
        # 2. 본문은 빼고 오직 메타데이터 딕셔너리만 변수에 할당
        pure_metadata = esg_data[target_index].get("metadata", {})
        
        print(f"📊 [공직자의 이해충돌 방지법] {target_index + 1}번째 청크의 메타데이터 전체")
        print("=" * 60)
        
        # 3. 줄바꿈과 들여쓰기(indent=4)를 적용해 한 글자도 빠짐없이 예쁘게 출력
        print(json.dumps(pure_metadata, indent=4, ensure_ascii=False))
        
        print("=" * 60)
    else:
        print(f"⚠️ 해당 인덱스({target_index})의 청크가 없습니다. 총 {len(esg_data)}개 존재합니다.")

except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다: {metadata_filepath}")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

📊 [공직자의 이해충돌 방지법] 60번째 청크의 메타데이터 전체
{
    "source": "KOTRA 임직원 행동강령.pdf",
    "origin_pdf": "KOTRA 임직원 행동강령",
    "page_num": "별표 11 부당한 업무지시의 판단기준",
    "page_start": 25,
    "page_end": 25,
    "chunk_id": "kotra_code_of_conduct_appendix_table_060",
    "document_name": "KOTRA 임직원 행동강령",
    "section_type": "appendix_table",
    "title": "별표 11 부당한 업무지시의 판단기준",
    "appendix": "별표 11",
    "content_format": "markdown",
    "is_deleted_article": false,
    "is_appendix": true,
    "is_table": true,
    "structure_key": "KOTRA 임직원 행동강령 > 별표 11 부당한 업무지시의 판단기준"
}


In [ ]:
kotra_code_of_conduct_article_004

In [36]:
# 메타 데이터 확인
import json

# 1. 경로 설정
metadata_filepath = "../../data/esg/json/KOTRA 임직원 행동강령.json"
target_index = 59  # 5번째 청크 (인덱스 4)

try:
    with open(metadata_filepath, "r", encoding="utf-8") as f:
        esg_data = json.load(f)
    
    if len(esg_data) > target_index:
        # 2. 본문은 빼고 오직 메타데이터 딕셔너리만 변수에 할당
        pure_metadata = esg_data[target_index].get("metadata", {})
        
        print(f"📊 [공직자의 이해충돌 방지법] {target_index + 1}번째 청크의 메타데이터 전체")
        print("=" * 60)
        
        # 3. 줄바꿈과 들여쓰기(indent=4)를 적용해 한 글자도 빠짐없이 예쁘게 출력
        print(json.dumps(pure_metadata, indent=4, ensure_ascii=False))
        
        print("=" * 60)
    else:
        print(f"⚠️ 해당 인덱스({target_index})의 청크가 없습니다. 총 {len(esg_data)}개 존재합니다.")

except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다: {metadata_filepath}")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

📊 [공직자의 이해충돌 방지법] 60번째 청크의 메타데이터 전체
{
    "source": "KOTRA 임직원 행동강령.pdf",
    "origin_pdf": "KOTRA 임직원 행동강령",
    "page_num": "별표 11 부당한 업무지시의 판단기준",
    "page_start": 25,
    "page_end": 25,
    "chunk_id": "kotra_code_of_conduct_appendix_table_060",
    "document_name": "KOTRA 임직원 행동강령",
    "section_type": "appendix_table",
    "title": "별표 11 부당한 업무지시의 판단기준",
    "appendix": "별표 11",
    "content_format": "markdown",
    "is_deleted_article": false,
    "is_appendix": true,
    "is_table": true,
    "structure_key": "KOTRA 임직원 행동강령 > 별표 11 부당한 업무지시의 판단기준"
}


In [33]:
'''
# 메타 데이터 유형 조회하고 싶을 때 
import json
from collections import Counter

# 1. 파일 경로 설정
metadata_filepath = "../../data/esg/json/KOTRA 임직원 행동강령.json"

try:
    with open(metadata_filepath, "r", encoding="utf-8") as f:
        esg_data = json.load(f)
    
    # 2. section_type의 개수를 세기 위한 카운터 객체 생성
    section_type_counts = Counter()
    
    # 3. 전체 청크를 순회하며 section_type 추출
    for chunk in esg_data:
        metadata = chunk.get("metadata", {})
        # section_type 키가 없을 경우를 대비해 기본값 "None" 설정
        s_type = metadata.get("section_type", "None(키 없음)")
        
        # 해당 타입의 카운트 1 증가
        section_type_counts[s_type] += 1

    # 4. 결과 출력
    print(f"📊 [KOTRA 임직원 행동강령] 총 {len(esg_data)}개 청크의 section_type 분포")
    print("=" * 50)
    
    # 개수가 많은 순서대로 정렬하여 출력
    for s_type, count in section_type_counts.most_common():
        print(f"🔹 {s_type}: {count}개")
        
    print("=" * 50)

except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다: {metadata_filepath}")
except Exception as e:
    print(f"❌ 오류 발생: {e}")
'''

📊 [KOTRA 임직원 행동강령] 총 64개 청크의 section_type 분포
🔹 article: 52개
🔹 deleted_article: 6개
🔹 appendix_table: 6개


In [50]:
import json
import re
from langchain_core.documents import Document
from langchain_pinecone import PineconeVectorStore

# =====================================================================
# [1단계] JSON 복원 함수 정의
# =====================================================================
def load_docs_from_json(input_path):
    """
    JSON 파일을 langchain_core.documents.Document 리스트로 복원.
    """
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    document_list = [
        Document(page_content=item["page_content"], metadata=item["metadata"])
        for item in data
    ]
    print(f"✅ {len(document_list)}개의 문서를 파일에서 성공적으로 불러왔습니다.")
    return document_list


# =====================================================================
# [2단계] ID 생성 및 조립 함수 정의 (🔥 3단 분기 및 3자리 Zero-padding 적용)
# =====================================================================
def extract_only_numbers(text):
    if text is None:
        return "0"
    text_str = str(text).strip()
    match = re.search(r'\d+', text_str)
    if match:
        return match.group(0)
    return "0"

def prepare_law_sequential_ids(docs, target_namespace):
    """
    사용자가 지정한 target_namespace를 기준으로,
    조항(article), 별표(appendix), 기타(etc)를 분류하여 3자리 순차 ID를 생성합니다.
    """
    if not docs:
        return []
    
    processed_ids = []
    chunk_counts_by_group = {}
    
    for doc in docs:
        article_info = doc.metadata.get("article_number")
        appendix_info = doc.metadata.get("appendix")
        
        # 1. 분기 처리 및 그룹 키 생성 (숫자는 무조건 3자리로 0 패딩)
        if article_info:
            clean_num = extract_only_numbers(article_info)
            group_key = f"{target_namespace}_{str(clean_num).zfill(3)}"
            
        elif appendix_info:
            clean_num = extract_only_numbers(appendix_info)
            group_key = f"{target_namespace}_appendix_{str(clean_num).zfill(3)}"
            
        else:
            # article, appendix 둘 다 없는 경우
            page_info = doc.metadata.get("page_num", "0")
            clean_num = extract_only_numbers(page_info)
            group_key = f"{target_namespace}_etc_{str(clean_num).zfill(3)}"
        
        # 2. 순번 카운팅
        if group_key not in chunk_counts_by_group:
            chunk_counts_by_group[group_key] = 1
        else:
            chunk_counts_by_group[group_key] += 1
            
        # 3. 최종 ID 조립 (순번 역시 3자리 0 패딩)
        seq_num = str(chunk_counts_by_group[group_key]).zfill(3)
        chunk_id = f"{group_key}_{seq_num}"
        processed_ids.append(chunk_id)
        
        # 4. 문서 메타데이터 내부 동기화
        doc.metadata["chunk_id"] = chunk_id
        
    return processed_ids


# =====================================================================
# [3단계] 🚀 파일명만 바꾸면 끝나는 범용 자동화 실행 영역
# 공직자의 이해충돌 방지법 / public_official_conflict_interest_act
# 부정청탁 및 금품등 수수의 금지에 관한 법률 / improper_solicitation_graft_act
# 직장 내 인권침해 예방지침 / workplace_human_rights_guideline
# KOTRA 임직원 이해충돌 방지제도 운영지침 / kotra_conflict_interest_guideline
# KOTRA 임직원 행동강령 / kotra_code_of_conduct
# =====================================================================
# 1. 처리할 JSON 파일 경로와 목적지 DB, 그리고 표에서 확정한 Namespace 지정
JSON_FILE_PATH = "../../data/esg/json/KOTRA 임직원 이해충돌 방지제도 운영지침.json"
TARGET_NAMESPACE = "kotra_conflict_interest_guideline"
CORRECT_INDEX_NAME = "finance-index-v2"

# 2. 문서 로드
current_docs = load_docs_from_json(JSON_FILE_PATH)

# 3. ID 세트 생성 및 내부 메타데이터 동기화
final_ids = prepare_law_sequential_ids(current_docs, target_namespace=TARGET_NAMESPACE)

print(f"🚪 파인콘에 생성될 Namespace: '{TARGET_NAMESPACE}'")
print(f"🎯 [ID 확인] 생성된 명찰 샘플 (상위 5개):")
for sample in final_ids[:5]:
    print(f"  - {sample}")

# 4. 파인콘 클라우드로 최종 적재
print(f"\n🚀 '{CORRECT_INDEX_NAME}' 창고의 '{TARGET_NAMESPACE}' Namespace로 Pinecone 빌드 시작...")
database_law = PineconeVectorStore.from_documents(
    documents=current_docs,        # 메타데이터가 치환된 문서 리스트
    embedding=embedding,
    index_name=CORRECT_INDEX_NAME,
    namespace=TARGET_NAMESPACE,
    ids=final_ids                  # 외부 식별 ID 배열
)
print(f"✅ 해당 문서의 모든 청크가 '{TARGET_NAMESPACE}' 방으로 전송 완료되었습니다! 🎉")

✅ 26개의 문서를 파일에서 성공적으로 불러왔습니다.
🚪 파인콘에 생성될 Namespace: 'kotra_conflict_interest_guideline'
🎯 [ID 확인] 생성된 명찰 샘플 (상위 5개):
  - kotra_conflict_interest_guideline_001_001
  - kotra_conflict_interest_guideline_002_001
  - kotra_conflict_interest_guideline_003_001
  - kotra_conflict_interest_guideline_004_001
  - kotra_conflict_interest_guideline_005_001

🚀 'finance-index-v2' 창고의 'kotra_conflict_interest_guideline' Namespace로 Pinecone 빌드 시작...
✅ 해당 문서의 모든 청크가 'kotra_conflict_interest_guideline' 방으로 전송 완료되었습니다! 🎉


In [42]:
'''
# 벡터 DB에서 특정 namespace 데이터 삭제
import os
from pinecone import Pinecone

# 1️⃣ 환경 세팅 (API Key 및 타겟 창고/방 이름 지정)
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
TARGET_INDEX_NAME = "finance-index-v2"  # 👈 데이터가 들어있는 진짜 창고 이름

# 인턴이 기존에 생성했던 부정청탁법의 Namespace 명칭
OLD_NAMESPACE = "kotra_code_of_conduct_article" 

# 2️⃣ 파인콘 클라이언트 접속
pc = Pinecone(api_key=pinecone_api_key)
index = pc.Index(TARGET_INDEX_NAME)

print(f"⏳ '{TARGET_INDEX_NAME}' 창고에서 '{OLD_NAMESPACE}' 방 데이터를 정밀 삭제 중...")

# 3️⃣ 🔥 [핵심] 해당 Namespace 전체 비우기
index.delete(delete_all=True, namespace=OLD_NAMESPACE)

print(f"✅ 삭제 완료! '{TARGET_INDEX_NAME}'에서 '{OLD_NAMESPACE}' 방이 깨끗하게 제거되었습니다.")

# 4️⃣ 삭제 후 현재 창고에 남은 방(Namespace) 목록 확인
print("\n🔄 현재 창고 잔여 데이터 현황:")
index_stats = index.describe_index_stats()
print(index_stats)
'''


SyntaxError: incomplete input (1009045861.py, line 1)

## 4. 문서 검색 및 답변
- 벡터DB에서 적합한 문서 찾고 LLM에 문서와 질의를 주면서 답변을 요청(RetrievalQA)
- RetrievalQA: 데이터를 검색(Retrieval)한 다음에 질문(Question)하고 답변(Answer) 할 것이다

In [52]:
# 문서를 가지고 왔으니까 질의를 해봐야 한다 
from langchain_openai import ChatOpenAI 

#llm = ChatOpenAI(model = 'gpt-4o')
llm = ChatOpenAI(model = 'gpt-5.1')

In [53]:
# Retrieval된 데이터는 LangChain에서 제공하는 프롬프트("rlm/rag-prompt") 사용해서 답변해보기
from langchain import hub 
from langchain.prompts import ChatPromptTemplate

# 기본 RAG 프롬프트 가져오기
base_prompt = hub.pull("rlm/rag-prompt")

# 시스템 역할 지시 추가해서 새 템플릿 만들기
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 재무팀에서 근무하는 해외무역관 정산 전문가입니다. "
               "[context]를 참고해서 사용자의 질문에 답변해주세요. "
               "[context]에 없는 정보는 지어내지 말고, 자료에 없음이라고 답하세요."
               ),
    *base_prompt.messages  # 기존 RAG 메시지 구조 유지
])

#### retrieve된 문서의 순서 변경하기
1. origin_pdf == "해외조직망정산지침"(A) 인 문서를 항상 앞으로 모은다.
2. 나머지 문서들(B 포함)은 뒤에 둔다.
3. 같은 그룹(A끼리, 그 외끼리) 는 원래 retriever가 준 순서를 그대로 유지한다.
4. A가 하나도 없는 경우는 기존 순서를 그대로 사용한다.

In [63]:
from typing import List, Any
from langchain_core.retrievers import BaseRetriever  # 버전에 따라 langchain.retrievers 일 수도 있음
from langchain.schema import Document

PRIORITY_ORIGIN = "정산지침"

class PriorityRetriever(BaseRetriever):
    base_retriever: Any

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> List[Document]:
        # 💡 Deprecation 경고를 해결하기 위해 get_relevant_documents 대신 invoke를 사용합니다.
        docs = self.base_retriever.invoke(query)
        docs.sort(
            key=lambda d: 0 if d.metadata.get("origin_pdf") == PRIORITY_ORIGIN else 1
        )
        return docs

    async def _aget_relevant_documents(self, query: str, *, run_manager=None) -> List[Document]:
        # 💡 비동기 역시 ainvoke로 변경하여 경고를 방지합니다.
        docs = await self.base_retriever.ainvoke(query)
        docs.sort(
            key=lambda d: 0 if d.metadata.get("origin_pdf") == PRIORITY_ORIGIN else 1
        )
        return docs

In [64]:
# 답변생성 (QA 체인 만들기)
#-- RetrievalQA를 통해 LLM에 전달
#-- RetrievalQA는 create_retrieval_chain으로 대체됨
#-- 실제 ChatBot 구현 시 create_retrieval_chain으로 변경하는 과정을 볼 수 있음

from langchain.chains import RetrievalQA 

base_retriever = database.as_retriever( # 벡터DB
    search_kwargs={"k": 3}) # 유사도로 몇 개 문서 찾을 것인지

priority_retriever = PriorityRetriever(base_retriever = base_retriever)

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = priority_retriever, # 위에서 만든 retriever (DB에서 문서 검색)
    chain_type_kwargs = {"prompt": prompt}, # 위에서 만든 prompt (정산 전문가)
    return_source_documents=True # 문서 소스 포함
)

In [65]:
query = '상품권 구매 시 정산은 어떻게 해야해?'

In [66]:
ai_message = qa_chain.invoke({"query": query})

# 답변 본문
answer = ai_message["result"]

In [67]:
answer

'자료에 상품권(기프티콘·상품권) 구매 시 정산 방법에 대한 별도 규정이나 사례는 없습니다.  \n다만 공통 원칙상 발생주의에 따라 ‘집행일 기준 해당 분기 내 정산’해야 하고, 집행경비를 여러 분기로 나눠 정산할 수 없으며, 회계연도 내에 미정산 시 불용·회수 대상이 된다는 점만 확인됩니다.'

In [60]:
ai_message["source_documents"]

[Document(id='5d545bec-e075-4348-ab82-b88f9b150d5f', metadata={'origin_pdf': '정산지침', 'page_num': '9조. 정산 시 유의사항 (공통사항)'}, page_content='9조.  정산 시 유의사항 (공통사항)3.  정산 시기 가. 자금정산은 발생주의 원칙에 의거, 집행일에 정산하여야 한다.  1) 차분기에 결제일이 도래할 분기말 카드사용분은 카드슬립과 영수증만으로 해당분기에 정산한다.   2) 카드대금 이외의 일반청구서도 회계연도 이월집행 및 정산이 불가하므로 연도말에 발급된 청구서는 수표발급과 함께 4/4분기에 정산하여야 하며, 4/4분기말 기준으로 동 자금이 인출되지 않았을 경우에는 자금잔액명세서 상 미결제수표 대금으로 차액을 설명하여야 한다.  3) 전화요금 등 공공요금의 경우도 12월에 발급된 청구서의 경우, 납기 마감일이 익년도일 경우라도 4/4분기에 해당요금을 지불하고 정산을 종료하여야 한다. 나. 계약조건에 의한 경우를 제외하고는 집행경비를 2개 이상의 분기에 분할하여 정산할 수 없다. 다. 모든 자금은 지원받은 당해 회계연도 내에 집행하고 정산하여야 하며, 연말까지 집행, 정산하지 아니하는 경우에는 불용액으로 처리, 회수하여야 한다.4.  지적사항에 대한 조치 가. 정산사항에 대한 지적, 보완, 참고사항 등을 통보받은 경우에는 성실한 내용으로 보고하여야 하며, 보고내용은 통보 도착일로부터 20일 이내에 본사에 도착하여야 한다. 나. 금액수정을 통보받은 경우에는 정산결과 통보공문 접수일을 기준으로 차액을 예금 또는 현금으로 입금하고, 해당 장부에 입금내역을 기장하여야 한다.5. 관련법규 및 지침 준수 회계관계직원의 책임에 관한 법률, 부정청탁 및 금품수수 등 금지에 관한 법률, 임직원행동강령 등 관련 법, 규정, 지침 등을 준수하고 이를 위반하는 행위를 하지 않도록 한다.'),
 Document(id='200a19f6-bb8c-4fa1-8992-5d6145539e42', me

In [61]:
import re
from pathlib import Path

# 강의에서는 위처럼 진행하지만 업데이트된 LangChain 문법은 `.invoke()` 활용을 권장 
# 객체에서는 속성에 점(.)을 사용하여 접근
# --- QA 실행 ---
ai_message = qa_chain.invoke({"query": query})

# 답변 본문
answer = ai_message["result"]

# -----------------------------
# 1) page_num 포맷 함수
# -----------------------------
def _format_page_token(page_val):
    """
    포맷 규칙:
    - 숫자(int) 또는 '7', '7.0', '07.00' 등 숫자 형태 → '7p'
    - 문자 섞인 경우 원문 그대로 유지 ('2조 목적' 등)
    """
    if page_val is None:
        return None

    # int → '3p'
    if isinstance(page_val, int):
        return f"{page_val}p"

    s = str(page_val).strip()
    if not s:
        return None

    # float 형태 문자열 ('7.0', '3.00') → int로 변환 후 p
    try:
        num = float(s)
        if num.is_integer():
            return f"{int(num)}p"
    except ValueError:
        pass

    # 순수 숫자 문자열 ('12') → '12p'
    if re.fullmatch(r"\d+", s):
        return f"{s}p"

    # 문자 섞임 ('2조 목적', 'p.12' 등) → 그대로
    return s

# -----------------------------
# 2) 순서 보존 중복 제거
# -----------------------------
def _dedup_preserve_order(items):
    seen = set()
    out = []
    for x in items:
        if x is None:
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

# -----------------------------
# 3) 파일명 정제
# -----------------------------
def _resolve_pdf_name(metadata: dict):
    name = (metadata or {}).get("origin_pdf") or (metadata or {}).get("source") or "알 수 없음"
    try:
        return Path(str(name)).name   # 경로에서 파일명만 추출
    except Exception:
        return str(name)

# -----------------------------
# 4) source_documents에서 출처 문자열 만들기
# -----------------------------
docs = ai_message.get("source_documents", []) or []

origins_order = []       # 파일 등장 순서
pages_by_origin = {}     # {파일명: [페이지토큰...]}
seen_pairs = set()       # (파일명, 페이지토큰) 중복 방지

# k=3이니까 상위 3개만 쓰고 싶으면 [:3], 2개만이면 [:2]
for d in docs[:3]:
    md = getattr(d, "metadata", {}) or {}
    origin = _resolve_pdf_name(md)
    page_token = _format_page_token(md.get("page_num"))
    if not page_token:
        continue

    pair = (origin, page_token)
    if pair in seen_pairs:
        continue
    seen_pairs.add(pair)

    if origin not in pages_by_origin:
        pages_by_origin[origin] = []
        origins_order.append(origin)

    if page_token not in pages_by_origin[origin]:
        pages_by_origin[origin].append(page_token)

# 최종 출처 문자열 구성
if origins_order:
    parts = []
    for origin in origins_order:
        pages = _dedup_preserve_order(pages_by_origin.get(origin, []))
        if not pages:
            continue
        pages_str = ", ".join(pages)
        parts.append(f"{pages_str} ({origin})")
    if parts:
        source_info = "📄출처: " + ", ".join(parts)
    else:
        source_info = "📄출처: 페이지 정보 없음"
else:
    source_info = "📄출처: 페이지 정보 없음"

# -----------------------------
# 5) 최종 출력
# -----------------------------
print(answer)
print(source_info)

상품권은 통상 ‘물품·서비스 구입을 위한 선급금(선결제)’ 성격이므로, **구입 시점(집행일)에 전액 정산**해야 합니다.  
다만 구체적인 계정과목(복리후생비, 판촉비 등)·증빙서류(거래명세, 사용내역, 수령자 관리대장 등)·예산사업코드 등 상품권 특화 지침은 제공된 자료에 없음입니다.
📄출처: 9조. 정산 시 유의사항 (공통사항) (정산지침), 40p, 41p (교육자료)


In [62]:
# 어떤 문서가 검색되었는지 확인
ai_message["source_documents"]

[Document(id='5d545bec-e075-4348-ab82-b88f9b150d5f', metadata={'origin_pdf': '정산지침', 'page_num': '9조. 정산 시 유의사항 (공통사항)'}, page_content='9조.  정산 시 유의사항 (공통사항)3.  정산 시기 가. 자금정산은 발생주의 원칙에 의거, 집행일에 정산하여야 한다.  1) 차분기에 결제일이 도래할 분기말 카드사용분은 카드슬립과 영수증만으로 해당분기에 정산한다.   2) 카드대금 이외의 일반청구서도 회계연도 이월집행 및 정산이 불가하므로 연도말에 발급된 청구서는 수표발급과 함께 4/4분기에 정산하여야 하며, 4/4분기말 기준으로 동 자금이 인출되지 않았을 경우에는 자금잔액명세서 상 미결제수표 대금으로 차액을 설명하여야 한다.  3) 전화요금 등 공공요금의 경우도 12월에 발급된 청구서의 경우, 납기 마감일이 익년도일 경우라도 4/4분기에 해당요금을 지불하고 정산을 종료하여야 한다. 나. 계약조건에 의한 경우를 제외하고는 집행경비를 2개 이상의 분기에 분할하여 정산할 수 없다. 다. 모든 자금은 지원받은 당해 회계연도 내에 집행하고 정산하여야 하며, 연말까지 집행, 정산하지 아니하는 경우에는 불용액으로 처리, 회수하여야 한다.4.  지적사항에 대한 조치 가. 정산사항에 대한 지적, 보완, 참고사항 등을 통보받은 경우에는 성실한 내용으로 보고하여야 하며, 보고내용은 통보 도착일로부터 20일 이내에 본사에 도착하여야 한다. 나. 금액수정을 통보받은 경우에는 정산결과 통보공문 접수일을 기준으로 차액을 예금 또는 현금으로 입금하고, 해당 장부에 입금내역을 기장하여야 한다.5. 관련법규 및 지침 준수 회계관계직원의 책임에 관한 법률, 부정청탁 및 금품수수 등 금지에 관한 법률, 임직원행동강령 등 관련 법, 규정, 지침 등을 준수하고 이를 위반하는 행위를 하지 않도록 한다.'),
 Document(id='200a19f6-bb8c-4fa1-8992-5d6145539e42', me

## 5. Retrieval을 위한 keyword 사전 활용
- 직장인이라는 질의가 들어오면, 직장인을 거주자로 자동으로 바꾸도록 설정
- Knowledge Base에서 사용되는 keyword를 활용하여 사용자 질문 수정
- LangChain Expression Language (LCEL)을 활용한 Chain 연계

In [51]:
query = '4직급의 기준을 자세하게 알려주세요. 보기좋게 연번으로 알려주세요.' # 표를 마크다운으로 바꾸면 더 잘 읽는지 확인

In [52]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

dictionary = ["기준을 나타내는 표현 -> 경력 기준"]

prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    그런 경우에는 질문만 리턴해주세요.
    사전: {dictionary}
    
    질문: {{question}}
""")

dictionary_chain = prompt | llm | StrOutputParser()
tax_chain = {"query": dictionary_chain} | qa_chain

In [53]:
new_question = dictionary_chain.invoke({"question": query})

In [54]:
# 바뀐 질의
new_question

'4직급의 경력 기준을 자세하게 알려주세요. 보기 좋게 연번으로 알려주세요.'

In [26]:
ai_response = tax_chain.invoke({"question": query})

In [28]:
print(ai_response['result'])

4직급의 경력 기준은 다음과 같습니다:

1. 행정 분야에서 근무한 6급 공무원으로 5년 이상 경력 소지자
2. 정부투자기관, 경제단체 및 유관기관에서 동일직급 2년 이상 경력 소지자
3. 소령 2년 이상 경력 소지자
4. 민간기업 과장급으로 유관부문 2년 이상 경력 소지자
5. 대학 및 전문학교 전임강사 2년 이상 경력 소지자
6. 전문조사기관의 연구원으로 3년 이상 경력 소지자
7. 해당 분야에 실무경력, 연구 또는 연수 경력자로 당해 직급에 자격이 있다고 인사위원회에서 인정되는 자
8. 기타 전항과 동등한 자격이 있다고 인사위원회에서 인정되는 자
